# MESACLIP + FOSI Sea Ice Area Validation - Timeseries
## Compare regional sea ice area (SIA) from MESACLIP and FOSI against NSIDC CDR obs, over the EngressNet ML regional domain

Adapted from `mesaclip_validate_sic_timeseries.ipynb` to (1) add FOSI alongside MESACLIP and (2) integrate SIA over
the actual ML regional domain (`DEFAULT_BBOX` in `functions_engressnet.py`: lat 60-80N, lon 170-220E) instead of the
whole Northern Hemisphere.

In [2]:
import numpy as np
import xarray as xr
from glob import glob
import matplotlib.pyplot as plt

## Regional domain
- must match the ML training/eval domain -- see `Version4/functions_engressnet.py:48` (`DEFAULT_BBOX`)

In [3]:
# lon in degrees_east, lat in degrees_north -- mirrors functions_engressnet.DEFAULT_BBOX
BBOX = {"lon_min": -190, "lon_max": -140, "lat_min": 60, "lat_max": 80}

def bbox_ij_slice(tlat, tlon, bbox):
    """Index-space (j, i) bounding box enclosing bbox on a curvilinear (2D lat/lon) grid.
    Longitude is normalized to [0, 360) on both sides of the comparison so a bbox crossing
    the dateline (like this one, 170E-220E) works the same as functions_engressnet.py's masking."""
    lon_min = bbox["lon_min"] % 360
    lon_max = bbox["lon_max"] % 360
    tlon360 = tlon % 360
    mask = (tlat >= bbox["lat_min"]) & (tlat <= bbox["lat_max"]) & (tlon360 >= lon_min) & (tlon360 <= lon_max)
    jj, ii = np.where(mask.values if hasattr(mask, "values") else mask)
    return slice(int(jj.min()), int(jj.max()) + 1), slice(int(ii.min()), int(ii.max()) + 1)

def fill_nan_coord(da):
    """MESACLIP/FOSI set TLAT/TLON themselves to NaN over all-land computational blocks
    (not just the data), which pcolormesh/contour reject as coordinate input. Since aice is
    NaN at exactly those same cells already (real land there, nothing to plot), it's safe to
    fill the coordinate gaps from nearby valid neighbors -- the mesh cell lands in roughly the
    right place but stays uncolored (NaN data), same as if TLAT/TLON had never been masked."""
    dims = da.dims
    return da.ffill(dims[0]).bfill(dims[0]).ffill(dims[1]).bfill(dims[1])

## Load MESACLIP data
- CESM-HR (ne120_t12) 9-member ensemble, monthly `aice` (%)
- historical 1920-2005 (`/gdex/data/d651007`) spliced with RCP8.5 2006-2100 (`/gdex/data/d651009`), matched by ensemble suffix
- pulled raw rather than from the old duvivier-preprocessed file, which lives in a `cgd`-group-only
  directory (`/glade/campaign/cgd/ppc/duvivier/...`) this account no longer has read access to

In [6]:
dir_hist = "/gdex/data/d651007"
dir_rcp85 = "/gdex/data/d651009"

hist_dirs = sorted(glob(f"{dir_hist}/*ihesp-hires*"))
rcp85_dirs = sorted(glob(f"{dir_rcp85}/*ihesp-hires*"))
# both dir names end in the same 3-digit ensemble suffix (e.g. "...002") -- match on that
member_ids = [d[-3:] for d in hist_dirs]
assert member_ids == [d[-3:] for d in rcp85_dirs], "historical/RCP8.5 member ordering does not match"
print("MESACLIP members:", member_ids)

MESACLIP members: ['002', '003', '004', '005', '006', '007', '008', '009', '010']


In [7]:
# open each member (historical + RCP8.5), crop to the regional bbox, then concat members.
# Cropping right after opening (while still dask-lazy) keeps this from ever materializing
# the full 3600x2400 global grid for 19 files/member.
mesa_members = []
nj_sl_mesa = ni_sl_mesa = None

for hdir, rdir in zip(hist_dirs, rcp85_dirs):
    hist_files = sorted(glob(f"{hdir}/ice/proc/tseries/month_1/*.aice.*.nc"))
    rcp85_files = sorted(glob(f"{rdir}/ice/proc/tseries/month_1/*.aice.*.nc"))
    ds_h = xr.open_mfdataset(hist_files, combine="by_coords")
    ds_r = xr.open_mfdataset(rcp85_files, combine="by_coords")
    if nj_sl_mesa is None:
        nj_sl_mesa, ni_sl_mesa = bbox_ij_slice(ds_h.TLAT, ds_h.TLON, BBOX)
    ds_h = ds_h[["aice", "tarea", "TLAT", "TLON"]].isel(nj=nj_sl_mesa, ni=ni_sl_mesa)
    ds_r = ds_r[["aice", "tarea", "TLAT", "TLON"]].isel(nj=nj_sl_mesa, ni=ni_sl_mesa)
    mesa_members.append(xr.concat([ds_h, ds_r], dim="time"))

ds_mesaclip = xr.concat(mesa_members, dim="member_id")
ds_mesaclip = ds_mesaclip.assign_coords(member_id=member_ids)
# TLAT/TLON are identical across members, so xr.concat's default compat check
# leaves them without a member_id dim at all (unlike aice, which does vary) --
# use them directly rather than trying to isel(member_id=0)
# also fill the NaN'd-out land-block coordinates so pcolormesh/contour can use them (see fill_nan_coord)
mesa_lat2d = fill_nan_coord(ds_mesaclip.TLAT)
mesa_lon2d = fill_nan_coord(ds_mesaclip.TLON)
ds_mesaclip

/glade/derecho/scratch/skygale/tmp/ipykernel_7190/1227130889.py:10: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_h = xr.open_mfdataset(hist_files, combine="by_coords")
/glade/derecho/scratch/skygale/tmp/ipykernel_7190/1227130889.py:11: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_r = xr.open_mfdataset(r

<xarray.Dataset> Size: 32GB
Dimensions:    (member_id: 9, time: 2172, nj: 434, ni: 474)
Coordinates:
  * member_id  (member_id) <U3 108B '002' '003' '004' ... '008' '009' '010'
  * time       (time) object 17kB 1920-02-01 00:00:00 ... 2101-01-01 00:00:00
    TLAT       (member_id, nj, ni) float32 7MB 60.2 60.2 60.2 ... 69.12 69.09
    TLON       (member_id, nj, ni) float32 7MB 164.7 164.8 164.9 ... 240.1 240.2
    ULON       (member_id, nj, ni) float32 7MB 164.7 164.9 ... -119.8 -119.8
    ULAT       (member_id, nj, ni) float32 7MB 60.24 60.23 60.23 ... 69.11 69.08
Dimensions without coordinates: nj, ni
Data variables:
    aice       (member_id, time, nj, ni) float32 16GB dask.array<chunksize=(1, 1, 434, 474), meta=np.ndarray>
    tarea      (member_id, time, nj, ni) float32 16GB dask.array<chunksize=(1, 120, 434, 474), meta=np.ndarray>
Attributes:
    title:        b.e13.BHISTC5.ne120_t12.cesm-ihesp-hires1.0.30-1920-2100.002
    contents:     Diagnostic and Prognostic Variables
    source:       sea ice model: Community Ice Code (CICE)
    comment:      All years have exactly 365 days
    comment2:     File written on model date 19200201
    comment3:     seconds elapsed into model date:      0
    conventions:  CF-1.0
    history:      This dataset was created on 2020-11-06 at 11:00

## Load FOSI data
- CESM-HR (t13), JRA55-forced, single realization, monthly `aice` (fraction 0-1)
- same native tripole grid as MESACLIP (confirmed identical TLAT/TLON), so shares the same crop indices

In [8]:
dir_fosi = "/glade/campaign/cgd/oce/projects/FOSI_BGC/HR/g.e22.TL319_t13.G1850ECOIAF_JRA_HR.4p2z.001/ice/proc/tseries/month_1"
fosi_files = sorted(glob(f"{dir_fosi}/*.aice.*.nc"))
ds_fosi = xr.open_mfdataset(fosi_files, combine="by_coords")

nj_sl_fosi, ni_sl_fosi = bbox_ij_slice(ds_fosi.TLAT, ds_fosi.TLON, BBOX)
ds_fosi = ds_fosi[["aice", "tarea", "TLAT", "TLON"]].isel(nj=nj_sl_fosi, ni=ni_sl_fosi)
fosi_lat2d = fill_nan_coord(ds_fosi.TLAT)
fosi_lon2d = fill_nan_coord(ds_fosi.TLON)
ds_fosi

/glade/derecho/scratch/skygale/tmp/ipykernel_7190/2127779694.py:3: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_fosi = xr.open_mfdataset(fosi_files, combine="by_coords")


<xarray.Dataset> Size: 1GB
Dimensions:  (time: 768, nj: 434, ni: 478)
Coordinates:
  * time     (time) object 6kB 1958-02-01 00:00:00 ... 2022-01-01 00:00:00
    TLAT     (nj, ni) float32 830kB dask.array<chunksize=(434, 478), meta=np.ndarray>
    TLON     (nj, ni) float32 830kB dask.array<chunksize=(434, 478), meta=np.ndarray>
    ULON     (nj, ni) float32 830kB dask.array<chunksize=(434, 478), meta=np.ndarray>
    ULAT     (nj, ni) float32 830kB dask.array<chunksize=(434, 478), meta=np.ndarray>
Dimensions without coordinates: nj, ni
Data variables:
    aice     (time, nj, ni) float32 637MB dask.array<chunksize=(1, 434, 478), meta=np.ndarray>
    tarea    (time, nj, ni) float32 637MB dask.array<chunksize=(12, 434, 478), meta=np.ndarray>
Attributes:
    title:             g.e22.TL319_t13.G1850ECOIAF_JRA_HR.4p2z.001
    contents:          Diagnostic and Prognostic Variables
    source:            Los Alamos Sea Ice Model (CICE) Version 5
    time_period_freq:  month_1
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    comment:           All years have exactly 365 days
    comment2:          File written on model date 19580201
    comment3:          seconds elapsed into model date:      0
    conventions:       CF-1.0
    history:           This dataset was created on 2022-07-17 at 19:38
    io_flavor:         io_pio

## Load CDR satellite SIC data
- daily NH data on 25km x 25km EASE grid
- crop to the bbox *before* resampling to monthly -- doing it after means averaging ~50x more
  data than needed and is dramatically slower

In [9]:
dir_grid = "/glade/campaign/cesm/development/pcwg/ssmi/"
ds_grid = xr.open_mfdataset(dir_grid + "grid_nh.nc", decode_times=False)
nj_sl_cdr, ni_sl_cdr = bbox_ij_slice(ds_grid.latitude, ds_grid.longitude, BBOX)

cdr_lat2d = ds_grid.latitude.isel(ygrid=nj_sl_cdr, xgrid=ni_sl_cdr)
cdr_lon2d = ds_grid.longitude.isel(ygrid=nj_sl_cdr, xgrid=ni_sl_cdr)

In [10]:
dir_in = "/glade/campaign/cesm/development/pcwg/ssmi/CDR/"
file_in = "cdr_seaice_conc_daily_nh.cdr.noleap.19790101-20211231.nc"
ds_cdr = xr.open_dataset(dir_in + file_in, chunks={"time": 365})

# reassign a proper noleap cftime time axis (the raw file's time coordinate isn't usable directly)
dates_noleap = xr.date_range(start="1979-01-01", end="2021-12-31", freq="D", calendar="noleap", use_cftime=True)
ds_cdr["time"] = dates_noleap

# grid_nh.nc's (ygrid, xgrid) and the data file's (jdim, idim) are the same native EASE grid
# in the same index order, just labeled differently -- so the bounding box computed above
# from the grid file applies directly here
sic_cdr = ds_cdr.cdr_seaice_conc_daily.isel(jdim=nj_sl_cdr, idim=ni_sl_cdr)

## Compute regional sea ice area (SIA)
- SIA = sum over the cropped domain of (ice concentration fraction x grid cell area)
- MESACLIP `aice` is in %, FOSI `aice` is a 0-1 fraction, CDR is in % -- normalize all to fraction before integrating
- `tarea` (MESACLIP/FOSI, native grid) is in m^2; CDR cells are a fixed 25km x 25km

In [11]:
# MESACLIP: aice is %, tarea in m^2 -> SIA in km^2, summed over the cropped domain per member/month
sia_mesaclip = ((ds_mesaclip.aice / 100) * ds_mesaclip.tarea).sum(dim=["nj", "ni"]) / 1e6

# FOSI: aice is already a 0-1 fraction
sia_fosi = (ds_fosi.aice * ds_fosi.tarea).sum(dim=["nj", "ni"]) / 1e6

In [12]:
sic_cdr_monthly = sic_cdr.resample(time="ME").mean()
# CDR is %; each cell is a fixed 25km x 25km EASE grid cell -> SIA in km^2
sia_cdr = ((sic_cdr_monthly / 100) * (25 * 25)).sum(dim=["jdim", "idim"])

### Get annual mean, March, and September

In [13]:
# MESACLIP
sia_mesaclip_mar = sia_mesaclip.sel(time=sia_mesaclip.time.dt.month == 3)
sia_mesaclip_sep = sia_mesaclip.sel(time=sia_mesaclip.time.dt.month == 9)
sia_mesaclip_ann = sia_mesaclip.resample(time="YE").mean()

# FOSI
sia_fosi_mar = sia_fosi.sel(time=sia_fosi.time.dt.month == 3)
sia_fosi_sep = sia_fosi.sel(time=sia_fosi.time.dt.month == 9)
sia_fosi_ann = sia_fosi.resample(time="YE").mean()

# CDR
sia_cdr_mar = sia_cdr.sel(time=sia_cdr.time.dt.month == 3)
sia_cdr_sep = sia_cdr.sel(time=sia_cdr.time.dt.month == 9)
sia_cdr_ann = sia_cdr.resample(time="YE").mean()

In [ ]:
# this cell does all the actual data reading for the full time series (all months, all years) --
# expect this to take several minutes, dominated by the 9-member MESACLIP read
sia_mesaclip_ann, sia_mesaclip_mar, sia_mesaclip_sep = (
    x.compute() for x in (sia_mesaclip_ann, sia_mesaclip_mar, sia_mesaclip_sep)
)
sia_fosi_ann, sia_fosi_mar, sia_fosi_sep = (
    x.compute() for x in (sia_fosi_ann, sia_fosi_mar, sia_fosi_sep)
)
sia_cdr_ann, sia_cdr_mar, sia_cdr_sep = (
    x.compute() for x in (sia_cdr_ann, sia_cdr_mar, sia_cdr_sep)
)

## Plot timeseries to compare

In [ ]:
import os
dir_out = "/glade/work/skygale/_projects/SeaIceDownscaling/Version4/saved_figs/mesaclip_fosi_validation/"
os.makedirs(dir_out, exist_ok=True)

In [ ]:
nens = len(sia_mesaclip_ann.member_id)
print("MESACLIP ensemble members:", nens)

# convert all to millions of km^2
sia_mesaclip_ann_m = sia_mesaclip_ann / 1e6
sia_mesaclip_mar_m = sia_mesaclip_mar / 1e6
sia_mesaclip_sep_m = sia_mesaclip_sep / 1e6

sia_fosi_ann_m = sia_fosi_ann / 1e6
sia_fosi_mar_m = sia_fosi_mar / 1e6
sia_fosi_sep_m = sia_fosi_sep / 1e6

sia_cdr_ann_m = sia_cdr_ann / 1e6
sia_cdr_mar_m = sia_cdr_mar / 1e6
sia_cdr_sep_m = sia_cdr_sep / 1e6

In [ ]:
def plot_panel(ax, mesa_da, fosi_da, cdr_da, title):
    # x-values are pulled from each series' own time coord (not a single shared array) --
    # annual resampling and month-selection don't always produce the same-length series
    # (e.g. CDR has a real multi-month sensor gap in 1987-88), so a shared x array can
    # silently mismatch a given y series' length
    xarr_mesa = mesa_da.time.dt.year.values
    xarr_fosi = fosi_da.time.dt.year.values
    xarr_cdr = cdr_da.time.dt.year.values
    for ii in range(nens):
        ax.plot(xarr_mesa, mesa_da.isel(member_id=ii), color="lightgrey", linestyle="-", linewidth=1.5, label="_nolegend_")
    ax.plot(xarr_mesa, mesa_da.mean(dim="member_id"), label=f"MESACLIP ensemble mean (n={nens})", color="black", linewidth=2)
    ax.plot(xarr_fosi, fosi_da, label="FOSI", color="darkorange", linewidth=2)
    ax.plot(xarr_cdr, cdr_da, label="NSIDC CDR", color="blue", linewidth=2)
    for yr in (1950, 2000, 2050):
        ax.axvline(x=yr, color="red", linestyle="--", linewidth=1.5)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("year", fontsize=13)
    ax.set_xlim([1920, 2100])
    ax.set_ylabel("Million km$^2$", fontsize=13)
    ax.tick_params(labelsize=12)

In [ ]:
# annual panel spans the full width, march/september share the row below --
# avoids the empty 4th quadrant a plain 2x2 grid left, and constrained_layout
# keeps the panels tight instead of the default wide gutters
fig = plt.figure(figsize=(13, 9), constrained_layout=True)
fout = "mesaclip_fosi_regional_timeseries_annual_mean"
gs = fig.add_gridspec(2, 2)

ax = fig.add_subplot(gs[0, :])
plot_panel(ax, sia_mesaclip_ann_m, sia_fosi_ann_m, sia_cdr_ann_m, "Annual Mean Regional SIA")
ax.legend(loc="lower left", fontsize=12, ncol=1)

ax = fig.add_subplot(gs[1, 0])
plot_panel(ax, sia_mesaclip_mar_m, sia_fosi_mar_m, sia_cdr_mar_m, "March Regional SIA")

ax = fig.add_subplot(gs[1, 1])
plot_panel(ax, sia_mesaclip_sep_m, sia_fosi_sep_m, sia_cdr_sep_m, "September Regional SIA")

fig.savefig(dir_out + fout + ".png", bbox_inches="tight", dpi=200)
